# Assignment High-Performance Computing for big data companies

## <span style="color:blue"> Setup </span>


In [1]:
from pyspark.sql import *
from pyspark import SparkContext, SparkConf
import time

spark = SparkSession.builder.master('local[3]').getOrCreate()

sc = spark.sparkContext

## <span style="color:blue"> Introduction </span>

This notebook presents a set of analyses on the NYC Yellow Taxi dataset as part of the course *High Performance Computing for Big Data Companies*, within the Master in Big Data Analytics program. The main objective of the assignment is to explore how Apache Spark can be applied to efficiently process large volumes of structured data in a distributed computing environment.

The dataset contains trip-level records for taxi rides in New York City, including information such as pickup and drop-off timestamps, trip distances, fares, tips, passenger counts, and payment types. Using Spark's core components — DataFrames, RDDs, and SQL — we carry out three different studies designed to demonstrate the flexibility and power of Spark in handling real-world big data workloads.

In addition to data exploration, this notebook evaluates the execution performance of Spark when using different degrees of parallelism, to assess how scalable the solution is as more computational resources (cores) are made available.


## <span style="color:blue"> Objectives and proposed studies </span>

The goals of this work are:

- To demonstrate the application of Spark’s APIs, including RDDs, DataFrames, and Spark SQL to perform meaningful data analyses on a large dataset.
- To measure and report the execution time and volume of data processed for each study.
- To analyze the speed-up achieved by increasing the number of processing cores, and evaluate the scalability of the implemented solutions.

We propose the following three studies:

1. **Average Speed by Hour of the Day**  
   Analyze how the average taxi speed varies throughout the day. This can help identify patterns in traffic congestion and trip efficiency.

2. **Most Frequent Taxi Trips**  
   Identify the most common pickup/drop-off location pairs. This study will be implemented using the **full Spark API**: RDDs (for initial transformations), DataFrames (for further structuring), and SQL (for querying).

3. **Analysis of Financial Variables**  
   Examine financial-related information such as total fares, tips, and passenger counts to detect high-revenue zones or time periods, and gain insights into rider behavior.

These studies will provide insights into both the dataset and the computational performance of Spark in processing it.

## <span style="color:blue"> Data description </span> 

In [2]:
 df = spark.read.format("parquet").option("inferSchema", "true").option("timestampFormat","yyyy-MM-dd HH:mm:ss").option("header", "true").option("mode", "DROPMALFORMED").load("yellow_tripdata_2025-01.parquet")

In [3]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       1| 2025-01-01 00:18:38|  2025-01-01 00:26:59|              1|          1.6|         1|                 N|         229|    

In [4]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



The dataset used in this assignment is part of the **NYC Yellow Taxi Trip Records**, provided by the New York City Taxi and Limousine Commission (TLC). It contains detailed records of individual taxi trips collected during **January 2025**.

The file is stored in **Parquet format**, which is optimized for distributed processing and efficient reading with Apache Spark.

Each row in the dataset corresponds to a single taxi ride and includes a wide range of features such as timestamps, locations, distances, fares, passenger counts, and payment details.

### <span style="color:blue"> Key Variables </span> 

Below is a summary of the most relevant fields in the dataset:

- `tpep_pickup_datetime`: Timestamp when the ride began.
- `tpep_dropoff_datetime`: Timestamp when the ride ended.
- `passenger_count`: Number of passengers in the vehicle.
- `trip_distance`: Distance traveled during the trip (in miles).
- `RatecodeID`: Rate code used for the trip (standard, JFK, Newark, etc.).
- `PULocationID`: Location ID where the passenger was picked up.
- `DOLocationID`: Location ID where the passenger was dropped off.
- `fare_amount`: Base fare charged for the trip.
- `extra`: Additional charges (e.g. congestion, rush hour).
- `mta_tax`: MTA tax collected.
- `tip_amount`: Tip given to the driver.
- `tolls_amount`: Toll charges incurred during the trip.
- `payment_type`: Method of payment (cash, credit card, etc.).
- `total_amount`: Total amount paid by the passenger.

## <span style="color:blue"> Code Logic </span> 


### <span style="color:blue"> First study: Average speed of taxis in terms of the hour </span>

### <span style="color:blue"> Second study: Most common trips </span>

Before performing any analysis, we conducted a preliminary data quality check and identified some records with inconsistent timestamps, specifically, cases where the pickup time is later than the dropoff time. Such entries are not logically valid and could negatively affect calculations related to trip duration and speed.

As a result, we applied a data curation step to remove all records where `tpep_pickup_datetime > tpep_dropoff_datetime`. This ensures that the dataset used for analysis contains only valid and reliable observations.

In [5]:
before = df.count() # showing numbers of rows before cleaning 
df_cleaned = df.filter(df["tpep_pickup_datetime"] <= df["tpep_dropoff_datetime"])
after = df_cleaned.count() # showing numbers of rows after cleaning 

print(f"Dropped {before - after} invalid rows")

Dropped 124 invalid rows


Now, we're able to perform our study.

In [6]:
## Using DataFrame transformations

start_time = time.time()

# Group by pickup and dropoff locations, count number of trips
trip_counts_df = df_cleaned.groupBy("PULocationID", "DOLocationID") \
    .count() \
    .orderBy("count", ascending=False)

# Show top 10 most common trips
trip_counts_df.show(10)

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.2f} seconds")

+------------+------------+-----+
|PULocationID|DOLocationID|count|
+------------+------------+-----+
|         237|         236|24839|
|         236|         237|21843|
|         236|         236|18221|
|         237|         237|17441|
|         161|         237|11463|
|         237|         161|10168|
|         161|         236| 9845|
|         239|         238| 9595|
|         142|         239| 9141|
|         239|         142| 8844|
+------------+------------+-----+
only showing top 10 rows
Execution time: 0.65 seconds


In [7]:
## Using Spark SQL

df_cleaned.createOrReplaceTempView("trips")

start_time = time.time()

# Use SQL to get most frequent pickup → dropoff pairs
most_common_trips_sql = spark.sql("""
    SELECT PULocationID, DOLocationID, COUNT(*) AS trip_count
    FROM trips
    GROUP BY PULocationID, DOLocationID
    ORDER BY trip_count DESC
    LIMIT 10
""")

most_common_trips_sql.show()

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.2f} seconds")

+------------+------------+----------+
|PULocationID|DOLocationID|trip_count|
+------------+------------+----------+
|         237|         236|     24839|
|         236|         237|     21843|
|         236|         236|     18221|
|         237|         237|     17441|
|         161|         237|     11463|
|         237|         161|     10168|
|         161|         236|      9845|
|         239|         238|      9595|
|         142|         239|      9141|
|         239|         142|      8844|
+------------+------------+----------+

Execution time: 0.39 seconds


In [8]:
## Using RDD

start_time = time.time()

# Convert DataFrame to RDD
trip_rdd = df_cleaned.rdd

# Map to ((pickup, dropoff), 1), then reduce by key
trip_pairs = trip_rdd.map(lambda row: ((row['PULocationID'], row['DOLocationID']), 1)) \
                     .reduceByKey(lambda x, y: x + y) \
                     .sortBy(lambda x: x[1], ascending=False)

# Show top 10
top_10_rdd = trip_pairs.take(10)

for ((pu, do), count) in top_10_rdd:
    print(f"Pickup: {pu}, Dropoff: {do} → {count} trips")

end_time = time.time()
execution_time = end_time - start_time
print(f"Execution time: {execution_time:.2f} seconds")

Pickup: 237, Dropoff: 236 → 24839 trips
Pickup: 236, Dropoff: 237 → 21843 trips
Pickup: 236, Dropoff: 236 → 18221 trips
Pickup: 237, Dropoff: 237 → 17441 trips
Pickup: 161, Dropoff: 237 → 11463 trips
Pickup: 237, Dropoff: 161 → 10168 trips
Pickup: 161, Dropoff: 236 → 9845 trips
Pickup: 239, Dropoff: 238 → 9595 trips
Pickup: 142, Dropoff: 239 → 9141 trips
Pickup: 239, Dropoff: 142 → 8844 trips
Execution time: 7.17 seconds


### <span style="color:blue"> Third study: Financial records (tips, persons, etc) </span> 

## <span style="color:blue"> Execution time and amount of data processed </span> 

### <span style="color:blue"> First study measures </span> 

### <span style="color:blue"> Second study measures </span> 

### <span style="color:blue"> Third study measures </span> 

## <span style="color:blue"> Speed-up study and scalability discussion </span> 

## <span style="color:blue"> Conclusions </span> 